In [ ]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np
from open_dataset_store import quick_start
import sys
sys.path.append("/home/jazz/Projects/DHCA-Framework/EnergyPlusSim")
from _5ZoneAutoDXVAV_zone_controller import ZoneController, get_w_from_rh
import math

def calculate_actual_flow(C, M):
    if C <= 0:
        return 0.0
    velocity = (9.3 + 0.008 * M) / (1.0 + math.exp(-0.16 * (C - 49.0 + 0.01 * M)))
    diameter = 0.110
    area = math.pi * (diameter / 2) ** 2
    air_density = 1.2 
    return velocity * area * air_density

# --- Global Design System ---
COLORS = {
    "Temp": "#2ecc71",       # Green
    "Hum": "#3498db",        # Blue
    "CO2": "#e67e22",        # Orange
    "Occ": "#9b59b6",        # Purple
    "Default": "#e74c3c",    # Red
    "Cool_SP": "#2ecc71",    # Green 1
    "Heat_SP": "#00bc8c",    # Green 2
}

def hex_to_rgba(hex_color, opacity=1.0):
    hex_color = hex_color.lstrip('#')
    return f"rgba({int(hex_color[0:2], 16)}, {int(hex_color[2:4], 16)}, {int(hex_color[4:6], 16)}, {opacity})"

def add_comfort_band(fig, row, col, x_arr, ymin, ymax):
    if np.isscalar(ymin):
        ymin = [ymin] * len(x_arr)
    if np.isscalar(ymax):
        ymax = [ymax] * len(x_arr)
        
    fig.add_trace(go.Scatter(
        x=x_arr, y=ymin, line=dict(color='rgba(255,255,255,0.4)', width=1, dash='dot'), 
        showlegend=False, hoverinfo='skip'
    ), row=row, col=col)
    
    fig.add_trace(go.Scatter(
        x=x_arr, y=ymax, fill='tonexty', fillcolor="rgba(255,255,255,0.05)", 
        line=dict(color='rgba(255,255,255,0.4)', width=1, dash='dot'), 
        showlegend=False, hoverinfo='skip'
    ), row=row, col=col)

def add_psychrometric_background(fig, row, col):
    T_range = np.linspace(10, 35, 100)
    P_atm = 101.325 # kPa
    for rh in [0.2, 0.4, 0.6, 0.8, 1.0]:
        P_sat = 0.61078 * np.exp(17.27 * T_range / (T_range + 237.3))
        P_v = rh * P_sat
        W = 0.622 * P_v / (P_atm - P_v)
        line_color = 'rgba(255, 255, 255, 0.2)' if rh == 1.0 else 'rgba(255, 255, 255, 0.05)'
        line_width = 1.5 if rh == 1.0 else 1
        fig.add_trace(go.Scatter(x=T_range, y=W, mode='lines', line=dict(color=line_color, width=line_width), hoverinfo='skip', showlegend=False), row=row, col=col)
        
        valid_idx = np.where((W <= 0.024) & (T_range <= 34))[0]
        if len(valid_idx) > 0:
            idx = valid_idx[-1]
            text_color = 'rgba(255, 255, 255, 0.5)' if rh == 1.0 else 'rgba(255, 255, 255, 0.3)'
            fig.add_trace(go.Scatter(
                x=[T_range[idx]], y=[W[idx]],
                mode='text',
                text=[f"{int(rh*100)}% RH"],
                textposition="top left",
                textfont=dict(color=text_color, size=10),
                showlegend=False, hoverinfo='skip'
            ), row=row, col=col)

def add_psychrometric_comfort(fig, row, col):
    T_cz = np.linspace(20, 25, 50)
    P_atm = 101.325
    P_sat = 0.61078 * np.exp(17.27 * T_cz / (T_cz + 237.3))
    W_30 = 0.622 * (0.3 * P_sat) / (P_atm - 0.3 * P_sat)
    W_60 = 0.622 * (0.6 * P_sat) / (P_atm - 0.6 * P_sat)
    
    cz_x = np.concatenate([T_cz, T_cz[::-1]])
    cz_y = np.concatenate([W_30, W_60[::-1]])
    
    fig.add_trace(go.Scatter(
        x=cz_x, y=cz_y, fill='toself', fillcolor='rgba(255,255,255,0.05)', 
        line=dict(color='rgba(255,255,255,0.4)', width=1, dash='dot'),
        showlegend=False, hoverinfo='skip'
    ), row=row, col=col)

def add_rh_co2_comfort(fig, row, col):
    cz_x = [30, 60, 60, 30, 30]
    cz_y = [0, 0, 1000, 1000, 0]
    fig.add_trace(go.Scatter(
        x=cz_x, y=cz_y, fill='toself', fillcolor='rgba(255,255,255,0.05)', 
        line=dict(color='rgba(255,255,255,0.4)', width=1, dash='dot'),
        showlegend=False, hoverinfo='skip'
    ), row=row, col=col)

def rh_to_w(T, rh):
    # Calculate absolute humidity W (kg/kg) from Temperature (C) and RH (%)
    rh_frac = rh / 100.0 if (rh > 1.0).any() else rh
    p_sat = 610.94 * np.exp(17.625 * T / (243.04 + T))
    p_vapor = rh_frac * p_sat
    return 0.62198 * p_vapor / (101325.0 - p_vapor)

# Initialize OpenDatasetStore and load data
store = quick_start('.', backend='local')
data_path = "raw_data/experiments/entry_0009_zone_002_1785989318_data.csv"
df = pd.read_csv(data_path)
df['timestamp'] = pd.to_datetime(df['timestamp'])

# Outlier Removal
co2_cols = ['outside_c', 'room_1_c', 'room_2_c', 'room_3_c', 'supply_c']
for col in co2_cols:
    if col in df.columns:
        invalid_mask = (df[col] < 400) | (df[col] > 10000)
        df.loc[invalid_mask, col] = np.nan
        df[col] = df[col].ffill().bfill()

# Feature Engineering
# Fix sensor issue: subtract 10 from all RH measurements
rh_cols = ['room_1_h', 'room_2_h', 'room_3_h', 'outside_h', 'supply_h']
for col in rh_cols:
    if col in df.columns:
        df[col] = df[col] - 10

df['room_1_w'] = rh_to_w(df['room_1_t'], df['room_1_h'])
df['avg_co2'] = (df['room_2_c'] + df['room_3_c']) / 2.0

# --- EKF Offline Evaluation Loop ---
print("Running EKF Offline Evaluation...")
zone_controller = ZoneController("Zone_1")
ekf_data = []
prev_time = None

class DummyLogger:
    def __init__(self): self.data = {}
    def add(self, k, v): self.data[k] = v

for i, row in df.iterrows():
    curr_time = row['timestamp'].timestamp()
    dt = 5.0 if prev_time is None else max(1.0, curr_time - prev_time)
    prev_time = curr_time
    
    T_out = row['outside_t']
    RH_out = row['outside_h'] / 100.0 if row['outside_h'] > 1.0 else row['outside_h']
    W_out = get_w_from_rh(T_out, RH_out)
    C_out = row['outside_c']
    
    T_in = row['room_1_t']
    RH_in = row['room_1_h'] / 100.0 if row['room_1_h'] > 1.0 else row['room_1_h']
    W_in = get_w_from_rh(T_in, RH_in)
    
    C_in = (row['room_2_c'] + row['room_3_c']) / 2 if row['room_3_c'] != 0 else row['room_2_c']
        
    T_s = row['supply_t']
    RH_s = row['supply_h'] / 100.0 if row['supply_h'] > 1.0 else row['supply_h']
    W_s = get_w_from_rh(T_s, RH_s)
    C_s = row['supply_c']
    
    vav_flow_real = calculate_actual_flow(row['fan'], row['mixer'])
    
    state_data = {
        'T_out': T_out, 'W_out': W_out, 'C_out': C_out,
        'T_in': T_in, 'W_in': W_in, 'C_in': C_in,
        'T_s': T_s, 'W_s': W_s, 'C_s': C_s,
        'temp_setpoint': 22.0, 
        'VAV_Flow': vav_flow_real
    }
    
    lg = DummyLogger()
    zone_controller.step(dt, state_data, lg)
    
    row_res = {'timestamp': row['timestamp']}
    ekf_states = ["EKF_x_T_in", "EKF_x_T_m", "EKF_x_W_in", "EKF_x_C_in", "EKF_x_d_T", "EKF_x_d_W", 
                  "EKF_x_N_occ", "EKF_x_alpha_ext", "EKF_x_alpha_int", "EKF_x_beta_air", "EKF_x_beta_mass", "EKF_x_d_C", "EKF_x_v_inf"]
    for idx, state_val in enumerate(zone_controller.x):
        row_res[f'Zone_1_{ekf_states[idx]}'] = state_val
        
    row_res['Zone_1_EKF_y_T_in'] = lg.data.get('EKF_y_T_in', np.nan)
    row_res['Zone_1_EKF_y_W_in'] = lg.data.get('EKF_y_W_in', np.nan)
    row_res['Zone_1_EKF_y_C_in'] = lg.data.get('EKF_y_C_in', np.nan)
    row_res['Zone_1_EKF_NIS'] = lg.data.get('EKF_NIS', np.nan)
    row_res['Zone_1_EKF_P_trace'] = lg.data.get('EKF_P_trace', np.nan)
    row_res['Zone_1_EKF_P_N_occ'] = zone_controller.P[6,6] if hasattr(zone_controller, 'P') else 0.0
    
    ekf_data.append(row_res)

ekf_df = pd.DataFrame(ekf_data)
df = pd.merge(df, ekf_df, on='timestamp', how='left')

if hasattr(zone_controller, 'P'):
    np.savetxt("final_P_Zone_1.csv", zone_controller.P, delimiter=",")
print("EKF Simulation Complete.")

# Filter to the requested time range
df = df.set_index('timestamp').between_time('19:00', '20:00').reset_index()


In [ ]:
# ==========================================
# View 1: Zone Controller Performance
# ==========================================
specs_v1 = [
    [{"type": "xy", "colspan": 2}, None], 
    [{"type": "xy", "colspan": 2}, None], 
    [{"type": "xy", "colspan": 2}, None],
    [{"type": "xy"}, {"type": "xy"}]
]
titles_v1 = [
    "Temperature Regulation (Room 1)",
    "Relative Humidity Regulation (Room 1)",
    "CO2 Monitoring (Avg Room 2 & 3)",
    "Psychrometric (Room 1)",
    "RH vs CO2 (Room 1 RH & Avg CO2)"
]

fig1 = make_subplots(rows=4, cols=2, subplot_titles=titles_v1, specs=specs_v1, vertical_spacing=0.08)

# Constant setpoint properties
T_SP = 22.0
T_MIN = T_SP - 1.0
T_MAX = T_SP + 1.0

time_arr = df['timestamp']

# 1.1 Temp (Room 1 vs Setpoint)
add_comfort_band(fig1, 1, 1, time_arr, T_MIN, T_MAX)
fig1.add_trace(go.Scatter(x=time_arr, y=df['room_1_t'], name="Room 1 Temp", line=dict(color=COLORS['Temp'], width=2)), row=1, col=1)
fig1.add_trace(go.Scatter(x=time_arr, y=df['outside_t'], name="Outdoor Temp", line=dict(color="rgba(255,255,255,0.4)", width=1, dash='dot')), row=1, col=1)
fig1.add_trace(go.Scatter(x=time_arr, y=df['supply_t'], name="Supply Temp", line=dict(color=hex_to_rgba(COLORS['Temp'], 0.6), width=1, dash='dash')), row=1, col=1)
fig1.add_trace(go.Scatter(x=time_arr, y=[T_SP]*len(time_arr), name="22°C Setpoint", line=dict(color="white", width=1, dash='dash')), row=1, col=1)

# 1.2 RH (Room 1 vs Comfort Band)
add_comfort_band(fig1, 2, 1, time_arr, 30, 60)
fig1.add_trace(go.Scatter(x=time_arr, y=df['room_1_h'], name="Room 1 RH (%)", line=dict(color=COLORS['Hum'], width=2)), row=2, col=1)
fig1.add_trace(go.Scatter(x=time_arr, y=df['outside_h'], name="Outdoor RH (%)", line=dict(color="rgba(255,255,255,0.4)", width=1, dash='dot')), row=2, col=1)
fig1.add_trace(go.Scatter(x=time_arr, y=df['supply_h'], name="Supply RH (%)", line=dict(color=hex_to_rgba(COLORS['Hum'], 0.6), width=1, dash='dash')), row=2, col=1)

# 1.3 CO2 (Avg 2 & 3 vs 1000ppm limit)
add_comfort_band(fig1, 3, 1, time_arr, 0, 1000)
fig1.add_trace(go.Scatter(x=time_arr, y=df['avg_co2'], name="Avg Room 2 & 3 CO2", line=dict(color=COLORS['CO2'], width=2)), row=3, col=1)
fig1.add_trace(go.Scatter(x=time_arr, y=df['outside_c'], name="Outdoor CO2 (ppm)", line=dict(color="rgba(255,255,255,0.4)", width=1, dash='dot')), row=3, col=1)
fig1.add_trace(go.Scatter(x=time_arr, y=df['supply_c'], name="Supply CO2 (ppm)", line=dict(color=hex_to_rgba(COLORS['CO2'], 0.6), width=1, dash='dash')), row=3, col=1)

# 1.4 Psychrometric (Room 1)
plasma_transparent = [
    [0.0, "rgba(0, 0, 0, 0)"],
    [0.001, "rgba(13, 8, 135, 0.0)"],
    [0.05, "rgba(156, 39, 176, 0.5)"],
    [0.15, "rgba(233, 30, 99, 0.8)"],
    [0.3, "rgba(255, 87, 34, 1)"],
    [0.5, "rgba(255, 193, 7, 1)"],
    [0.7, "rgba(255, 235, 59, 1)"],
    [1.0, "rgba(255, 255, 255, 1)"]
]

add_psychrometric_background(fig1, 4, 1)
add_psychrometric_comfort(fig1, 4, 1)
fig1.add_trace(go.Histogram2d(
    x=df['room_1_t'], y=df['room_1_w'],
    colorscale=plasma_transparent, showscale=False,
    nbinsx=60, nbinsy=60
), row=4, col=1)

# 1.5 RH vs CO2
add_rh_co2_comfort(fig1, 4, 2)
fig1.add_trace(go.Histogram2d(
    x=df['room_1_h'], y=df['avg_co2'],
    colorscale=plasma_transparent, showscale=True,
    colorbar=dict(title="Frequency", x=1.05, y=0.1, len=0.2),
    nbinsx=60, nbinsy=60
), row=4, col=2)

fig1.update_xaxes(range=[10, 35], title_text="Temperature (°C)", row=4, col=1)
fig1.update_yaxes(range=[0, 0.025], title_text="Abs Hum (kg/kg)", row=4, col=1)

fig1.update_xaxes(title_text="Relative Humidity (%)", row=4, col=2)
fig1.update_yaxes(title_text="CO2 (ppm)", row=4, col=2)

fig1.update_layout(
    height=1400, template="plotly_dark", title_text="Zone Controller Deep Dive",
    margin=dict(l=30, r=30, t=80, b=30),
)

fig1.show()


In [ ]:
# ==========================================
# View 2: AHU Coordinator
# ==========================================
fig2 = make_subplots(
    rows=3, cols=1,
    subplot_titles=["Temperature Arbitration (Ideal vs Supply)", "Humidity Arbitration", "CO2 Arbitration"],
    vertical_spacing=0.1
)

# 2.1 Temp
add_comfort_band(fig2, 1, 1, time_arr, T_MIN, T_MAX)
fig2.add_trace(go.Scatter(x=time_arr, y=df['zone_ideal_temp'], name="Zone Ideal Temp Ask", line=dict(color=hex_to_rgba(COLORS['Cool_SP'], 0.5), width=2, dash='dot')), row=1, col=1)
fig2.add_trace(go.Scatter(x=time_arr, y=df['supply_t'], name="Actual Supply Temp", line=dict(color=COLORS['Cool_SP'], width=2.5)), row=1, col=1)

# 2.2 Humidity
fig2.add_trace(go.Scatter(x=time_arr, y=df['zone_ideal_hum'] * 100 if df['zone_ideal_hum'].max() <= 1.0 else df['zone_ideal_hum'], name="Zone Ideal Hum Ask", line=dict(color=hex_to_rgba(COLORS['Hum'], 0.5), width=2, dash='dot')), row=2, col=1)
fig2.add_trace(go.Scatter(x=time_arr, y=df['supply_h'], name="Actual Supply Hum", line=dict(color=COLORS['Hum'], width=2.5)), row=2, col=1)

# 2.3 CO2
fig2.add_trace(go.Scatter(x=time_arr, y=df['zone_ideal_co2'], name="Zone Ideal CO2 Ask", line=dict(color=hex_to_rgba(COLORS['CO2'], 0.5), width=2, dash='dot')), row=3, col=1)
fig2.add_trace(go.Scatter(x=time_arr, y=df['supply_c'], name="Actual Supply CO2", line=dict(color=COLORS['CO2'], width=2.5)), row=3, col=1)

fig2.update_layout(
    height=800, template="plotly_dark", title_text="AHU Coordinator Arbitration",
    margin=dict(l=30, r=30, t=80, b=30),
)

fig2.show()


In [ ]:
# ==========================================
# View 3: AHU Actuation (Commands vs Hardware)
# ==========================================
fig3 = make_subplots(
    rows=4, cols=1,
    subplot_titles=["Fan Command", "Mixer Ratio", "Cooler Command", "Heater Command"],
    vertical_spacing=0.08
)

# 3.1 Fan
fig3.add_trace(go.Scatter(x=time_arr, y=df['fan_cmnd'], name="Controller Fan Cmd %", line=dict(color=COLORS['Default'], width=2)), row=1, col=1)
fig3.add_trace(go.Scatter(x=time_arr, y=df['fan'], name="Hardware Fan Actual %", line=dict(color=hex_to_rgba(COLORS['Default'], 0.4), width=3)), row=1, col=1)

# 3.2 Mixer
fig3.add_trace(go.Scatter(x=time_arr, y=df['mixer_ratio'], name="Controller Mixer Cmd %", line=dict(color=COLORS['Hum'], width=2)), row=2, col=1)
fig3.add_trace(go.Scatter(x=time_arr, y=df['mixer'], name="Hardware Mixer Actual %", line=dict(color=hex_to_rgba(COLORS['Hum'], 0.4), width=3)), row=2, col=1)

# 3.3 Cooler
fig3.add_trace(go.Scatter(x=time_arr, y=df['cooler_cmd'], name="Controller Cooler Cmd", line=dict(color=COLORS['Hum'], width=2)), row=3, col=1)
if 'coolerState' in df.columns:
    fig3.add_trace(go.Scatter(x=time_arr, y=df['coolerState'], name="Hardware Cooler State", line=dict(color=hex_to_rgba(COLORS['Hum'], 0.4), width=3)), row=3, col=1)

# 3.4 Heater
fig3.add_trace(go.Scatter(x=time_arr, y=df['heater_cmd'], name="Controller Heater Cmd", line=dict(color=COLORS['Default'], width=2)), row=4, col=1)
if 'heaterState' in df.columns:
    fig3.add_trace(go.Scatter(x=time_arr, y=df['heaterState'], name="Hardware Heater State", line=dict(color=hex_to_rgba(COLORS['Default'], 0.4), width=3)), row=4, col=1)

fig3.update_layout(
    height=1000, template="plotly_dark", title_text="AHU Component Actuation",
    margin=dict(l=30, r=30, t=80, b=30),
)

fig3.show()


In [ ]:
# ==========================================
# View 4: EKF State Estimation
# ==========================================
specs_ekf = [
    [{"colspan": 3, "type": "xy"}, None, None],
    [{"type": "xy"}, {"type": "xy"}, {"type": "xy"}],
    [{"colspan": 3, "type": "xy"}, None, None],
    [{"colspan": 2, "type": "xy"}, None, {"type": "heatmap"}]
]
titles_ekf = [
    "Occupancy Estimation", "Temperature Residual", "Absolute Humidity Residual", "CO2 Residual", 
    "Normalized Innovation Squared (NIS)", 
    "Covariance Convergence (Trace P)", "Final State Correlation Matrix"
]

fig4 = make_subplots(rows=4, cols=3, subplot_titles=titles_ekf, specs=specs_ekf, vertical_spacing=0.08)

# 4.1 Occupancy
fig4.add_trace(go.Scatter(x=time_arr, y=df['Zone_1_EKF_x_N_occ'], name="Est Occ", line=dict(color=COLORS['Occ'], width=1.5, dash='dot')), row=1, col=1)

if 'Zone_1_EKF_P_N_occ' in df.columns:
    std_dev = np.sqrt(df['Zone_1_EKF_P_N_occ'].clip(lower=0))
    upper = df['Zone_1_EKF_x_N_occ'] + 2*std_dev
    lower = df['Zone_1_EKF_x_N_occ'] - 2*std_dev
    fig4.add_trace(go.Scatter(x=time_arr, y=upper, mode='lines', line=dict(width=0), showlegend=False), row=1, col=1)
    fig4.add_trace(go.Scatter(x=time_arr, y=lower, mode='lines', line=dict(width=0), fill='tonexty', fillcolor=hex_to_rgba(COLORS['Occ'], 0.15), name="±2σ Band"), row=1, col=1)

# 4.2 Residuals
res_vars = [('Zone_1_EKF_y_T_in', 'T_in', COLORS['Temp'], 1), 
            ('Zone_1_EKF_y_W_in', 'W_in', COLORS['Hum'], 2), 
            ('Zone_1_EKF_y_C_in', 'C_in', COLORS['CO2'], 3)]
for col_name, label, color, col_idx in res_vars:
    if col_name in df.columns:
        fig4.add_trace(go.Violin(x=df[col_name], name=label, orientation='h', line_color=color, meanline_visible=True), row=2, col=col_idx)

# 4.3 NIS
if 'Zone_1_EKF_NIS' in df.columns:
    fig4.add_trace(go.Scatter(x=time_arr, y=df['Zone_1_EKF_NIS'], name="NIS", line=dict(color="#f1c40f", width=1.5)), row=3, col=1)
    add_comfort_band(fig4, 3, 1, time_arr, 0.216, 7.815)
    fig4.update_yaxes(type="log", row=3, col=1)

# 4.4 Covariance Convergence
if 'Zone_1_EKF_P_trace' in df.columns:
    fig4.add_trace(go.Scatter(x=time_arr, y=df['Zone_1_EKF_P_trace'], name="Trace(P)", line=dict(color="#f1c40f", width=2)), row=4, col=1)
    fig4.update_yaxes(type="log", row=4, col=1)

# 4.5 Correlation Heatmap
import os
p_csv = "final_P_Zone_1.csv"
if os.path.exists(p_csv):
    try:
        p_mat = np.loadtxt(p_csv, delimiter=",")
        d = np.sqrt(np.diag(p_mat))
        corr_mat = p_mat / np.outer(d, d)
        corr_mat = np.clip(corr_mat, -1, 1)
        states = ["T_in", "T_m", "W_in", "C_in", "d_T", "d_W", "N_occ", "alpha_ext", "alpha_int", "beta_air", "beta_mass", "d_C", "v_inf"]
        
        if len(states) == p_mat.shape[0]:
            fig4.add_trace(go.Heatmap(
                z=corr_mat[::-1], x=states, y=states[::-1], 
                colorscale="PuOr", zmin=-1, zmax=1, 
                colorbar=dict(title="Correlation", len=0.2, y=0.1)
            ), row=4, col=3)
            fig4.update_yaxes(scaleanchor="x", scaleratio=1, row=4, col=3)
    except Exception as e: 
        print(f"Heatmap Error: {e}")

fig4.update_layout(
    height=1600, template="plotly_dark", title_text="Zone 1 EKF Filter Diagnostics",
    margin=dict(l=30, r=30, t=80, b=30),
)

fig4.show()
